In [2]:
import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

In [107]:
from transformers import AlbertModel, AlbertConfig

# Membuat model dengan konfigurasi
config = AlbertConfig.from_pretrained('indobenchmark/indobert-lite-base-p2')
model1 = AlbertModel(config)

# Memuat weight dari pretrained model
pretrained_model = AlbertModel.from_pretrained('indobenchmark/indobert-lite-base-p2')

# Menyalin state_dict dari pretrained model ke model yang baru dibuat
model1.load_state_dict(pretrained_model.state_dict())

# Verifikasi weight sudah sama
print(model1.embeddings.LayerNorm.weight)

Parameter containing:
tensor([0.8965, 1.3851, 1.3848, 1.4385, 1.4131, 1.3894, 1.2970, 1.5432, 1.4673,
        1.4771, 1.6056, 1.4827, 1.5077, 1.4535, 1.4892, 1.4622, 1.4723, 1.5928,
        1.5936, 1.2219, 1.4651, 1.5027, 1.1348, 1.4652, 0.9018, 1.3907, 1.4780,
        1.4864, 1.3379, 1.3838, 1.6363, 1.5410, 1.5089, 1.5027, 1.5258, 1.4884,
        1.5282, 1.4625, 1.4915, 1.4466, 1.5284, 1.4329, 0.9101, 1.4773, 1.4578,
        1.4556, 1.5459, 1.4726, 1.5137, 1.4828, 1.3085, 1.0074, 0.8707, 1.4524,
        1.4169, 1.4939, 1.4417, 1.4708, 1.4425, 1.5908, 1.4455, 1.4174, 1.4451,
        1.4574, 1.4688, 1.4537, 1.4693, 1.6129, 1.3694, 1.4665, 1.1254, 1.4958,
        1.6083, 1.4973, 1.4159, 1.5163, 1.4512, 1.5002, 1.5398, 0.9201, 1.4796,
        1.4241, 1.3584, 1.4683, 1.5242, 0.8194, 1.4904, 1.4488, 1.4664, 1.4880,
        1.4673, 1.4191, 1.5021, 1.4128, 1.3807, 0.9933, 1.4531, 1.4512, 1.4687,
        1.4626, 1.3956, 1.4076, 1.4994, 1.4872, 1.5519, 1.4390, 1.5148, 1.5036,
        1.5074, 1.

# RMSNORM

In [124]:
from transformers import AlbertConfig, AlbertModel
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * (x / norm)

# Ganti semua LayerNorm di BERT dengan RMSNorm
def bert_rmsnorm(bert_model, use_pretrained_weight=True):
    for name, module in bert_model.named_children():
        if(isinstance(module, nn.LayerNorm)):
            # create rms norm
            # print("pretrained weight module", module.weight)
            hidden_size = module.normalized_shape[0]
            rms_norm = RMSNorm(hidden_size, eps=module.eps)
            # print("rms norm weight (before) ", rms_norm.weight)
            # copy weight from pretrained
            if use_pretrained_weight:
                with torch.no_grad():
                    rms_norm.weight.copy_(module.weight)
            # print("rms norm weight (after) ", rms_norm.weight)
            setattr(bert_model, name, rms_norm)
        else:
            bert_rmsnorm(module, use_pretrained_weight)

    return bert_model
                
config = AlbertConfig.from_pretrained('indobenchmark/indobert-lite-base-p2')
model = AlbertModel(config)
pretrained_model = AlbertModel.from_pretrained('indobenchmark/indobert-lite-base-p2')
model.load_state_dict(pretrained_model.state_dict())
model = bert_rmsnorm(model)
print("embedding", model.embeddings.LayerNorm.weight)
# print("encoder att", model.encoder.layer[0].attention.output.LayerNorm.weight)
# print("encoder output", model.encoder.layer[0].output.LayerNorm.weight)
model

embedding Parameter containing:
tensor([0.8965, 1.3851, 1.3848, 1.4385, 1.4131, 1.3894, 1.2970, 1.5432, 1.4673,
        1.4771, 1.6056, 1.4827, 1.5077, 1.4535, 1.4892, 1.4622, 1.4723, 1.5928,
        1.5936, 1.2219, 1.4651, 1.5027, 1.1348, 1.4652, 0.9018, 1.3907, 1.4780,
        1.4864, 1.3379, 1.3838, 1.6363, 1.5410, 1.5089, 1.5027, 1.5258, 1.4884,
        1.5282, 1.4625, 1.4915, 1.4466, 1.5284, 1.4329, 0.9101, 1.4773, 1.4578,
        1.4556, 1.5459, 1.4726, 1.5137, 1.4828, 1.3085, 1.0074, 0.8707, 1.4524,
        1.4169, 1.4939, 1.4417, 1.4708, 1.4425, 1.5908, 1.4455, 1.4174, 1.4451,
        1.4574, 1.4688, 1.4537, 1.4693, 1.6129, 1.3694, 1.4665, 1.1254, 1.4958,
        1.6083, 1.4973, 1.4159, 1.5163, 1.4512, 1.5002, 1.5398, 0.9201, 1.4796,
        1.4241, 1.3584, 1.4683, 1.5242, 0.8194, 1.4904, 1.4488, 1.4664, 1.4880,
        1.4673, 1.4191, 1.5021, 1.4128, 1.3807, 0.9933, 1.4531, 1.4512, 1.4687,
        1.4626, 1.3956, 1.4076, 1.4994, 1.4872, 1.5519, 1.4390, 1.5148, 1.5036,
        

AlbertModel(
  (embeddings): AlbertEmbeddings(
    (word_embeddings): Embedding(30000, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): RMSNorm()
    (dropout): Dropout(p=0, inplace=False)
  )
  (encoder): AlbertTransformer(
    (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
    (albert_layer_groups): ModuleList(
      (0): AlbertLayerGroup(
        (albert_layers): ModuleList(
          (0): AlbertLayer(
            (full_layer_layer_norm): RMSNorm()
            (attention): AlbertSdpaAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (attention_dropout): Dropout(p=0, inplace=False)
              (output_dropout): Dropout(p=0, inplace=False)
              (dense): Linear(

# ADANORM

In [ ]:
import torch
import torch.nn as nn

class AdaNorm(nn.Module):
    def __init__(self, num_features, eps=0, C=1.0, k=1/10):
        super(AdaNorm, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.C = C
        self.k = k

    def forward(self, x):
        mean = torch.mean(x, dim=-1, keepdim=True)
        variance = torch.var(x, dim=-1, keepdim=True, unbiased=False)
        std = torch.sqrt(variance + self.eps)

        # Normalize
        y = (x - mean) / std

        # Adaptive transformation function
        phi_y = self.C * (1 - self.k * y)

        # Detach the gradient of phi_y
        phi_y = phi_y.detach()

        # Final output
        z = phi_y * y

        return z
    
def bert_adanorm(bert_model, use_pretrained_weight=True):
    for name, module in bert_model.named_children():
        if(isinstance(module, nn.LayerNorm)):
            # create rms norm
            # print("pretrained weight module", module.weight)
            hidden_size = module.normalized_shape[0]
            adanorm = AdaNorm(hidden_size, eps=0, C=1, k=1/10)
            setattr(bert_model, name, adanorm)
        else:
            bert_adanorm(module, use_pretrained_weight)

    return bert_model
                
config = AlbertConfig.from_pretrained('indobenchmark/indobert-lite-base-p2')
model = AlbertModel(config)
pretrained_model = AlbertModel.from_pretrained('indobenchmark/indobert-lite-base-p2')
model.load_state_dict(pretrained_model.state_dict())
model = bert_adanorm(model)
model

embedding AdaNorm()


AlbertModel(
  (embeddings): AlbertEmbeddings(
    (word_embeddings): Embedding(30000, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): AdaNorm()
    (dropout): Dropout(p=0, inplace=False)
  )
  (encoder): AlbertTransformer(
    (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
    (albert_layer_groups): ModuleList(
      (0): AlbertLayerGroup(
        (albert_layers): ModuleList(
          (0): AlbertLayer(
            (full_layer_layer_norm): AdaNorm()
            (attention): AlbertSdpaAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (attention_dropout): Dropout(p=0, inplace=False)
              (output_dropout): Dropout(p=0, inplace=False)
              (dense): Linear(

# FILTER RESPONSE NORM

In [ ]:
class FilterResponseNormNd(nn.Module):
    def __init__(self, hidden_dim, eps=1e-6, learnable_eps=False):
        super(FilterResponseNormNd, self).__init__()
        self.eps = nn.Parameter(torch.ones(1, hidden_dim, 1) * eps)
        if not learnable_eps:
            self.eps.requires_grad_(False)
        self.gamma = nn.Parameter(torch.ones(1, hidden_dim, 1))
        self.beta = nn.Parameter(torch.zeros(1, hidden_dim, 1))
        self.tau = nn.Parameter(torch.zeros(1, hidden_dim, 1))
    
    def forward(self, x):
        # nu2 = torch.mean(x**2, dim=[1, 2], keepdim=True)
        nu2 = torch.mean(x**2, dim=-1, keepdim=True)
        x = x * torch.rsqrt(nu2 + torch.abs(self.eps))
        return torch.max(self.gamma * x + self.beta, self.tau)

def bert_frnorm(bert_model, use_pretrained_weight=False):
    for name, module in bert_model.named_children():
        if(isinstance(module, nn.LayerNorm)):
            hidden_size = module.normalized_shape[0]
            fr_norm = FilterResponseNormNd(hidden_dim=hidden_size, eps=module.eps)
            if use_pretrained_weight:
                with torch.no_grad():
                    fr_norm.gamma.copy_(module.weight.view_as(fr_norm.gamma))
                    fr_norm.beta.copy_(module.bias.view_as(fr_norm.beta))
                    fr_norm.tau.fill_(0)
            setattr(bert_model, name, fr_norm)
        else:
            bert_frnorm(module, use_pretrained_weight)

    return bert_model
                
config = AlbertConfig.from_pretrained('indobenchmark/indobert-lite-base-p2')
model = AlbertModel(config)
pretrained_model = AlbertModel.from_pretrained('indobenchmark/indobert-lite-base-p2')
model.load_state_dict(pretrained_model.state_dict())
model = bert_frnorm(model)
model

embedding Parameter containing:
tensor([[[1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
         [1.],
        

AlbertModel(
  (embeddings): AlbertEmbeddings(
    (word_embeddings): Embedding(30000, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): FilterResponseNormNd()
    (dropout): Dropout(p=0, inplace=False)
  )
  (encoder): AlbertTransformer(
    (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
    (albert_layer_groups): ModuleList(
      (0): AlbertLayerGroup(
        (albert_layers): ModuleList(
          (0): AlbertLayer(
            (full_layer_layer_norm): FilterResponseNormNd()
            (attention): AlbertSdpaAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (attention_dropout): Dropout(p=0, inplace=False)
              (output_dropout): Dropout(p=0, inplace=False)
    

# ADD NORM LAYER

In [14]:
import torch
import torch.nn as nn
from transformers.models.bert.modeling_bert import BertModel, BertConfig

class CustomBertPooler(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)  # Tambahkan LayerNorm
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.activation = nn.Tanh()

    def forward(self, hidden_states):
        # Ambil token CLS (hidden_states[:, 0])
        pooled_output = hidden_states[:, 0]  
        
        # Tambahkan normalisasi sebelum linear
        pooled_output = self.norm(pooled_output)  # LayerNorm sebelum masuk ke linear
        
        # Linear + Activation
        pooled_output = self.dense(pooled_output)
        pooled_output = self.activation(pooled_output)
        
        return pooled_output

# Custom BERT dengan Pooler yang Dimodifikasi
class BertWithCustomPooler(BertModel):
    def __init__(self, config):
        super().__init__(config)
        self.pooler = CustomBertPooler(config)  # Gunakan pooler yang telah dimodifikasi

# Load model dengan konfigurasi
config = BertConfig()
model = BertWithCustomPooler(config)

# Cek apakah Pooler sudah memiliki LayerNorm
print(model)


BertWithCustomPooler(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, in